# Integrated oligodendrocyte atlas: is the C4b⁺ oligodendrocyte one state or several, and what does it depend on?

Reads the atlas built by `build_oligodendrocyte_atlas.ipynb` (twelve datasets, mouse and human, scVI/scANVI integrated) and asks:

1. **Integration and shared states.** Do oligodendrocytes from different datasets and species mix, and is there a C4b-high region of the embedding that all datasets contribute to?
2. **Label transfer.** Lerma-Martin's human disease-associated oligodendrocyte subtypes (`OL_Dis*`) were the only seeded labels. Where do mouse aging, AD and demyelination C4b⁺ cells land, and does the `OL_Dis` probability track disease / age in each dataset?
3. **Conserved markers.** Differential expression of the C4b-high clusters vs homeostatic clusters *within each dataset*, then the overlap of top genes across datasets: the cross-species core of the state.
4. **Gene programs.** Per-dataset NMF programs matched across datasets by cosine similarity: which programs recur, which contain C4b / Serpina3n / MHC-I / complement regulators, and how their usage changes with condition.
5. **Genetic dependency.** Three knock-outs already present in the data: Rag1-KO (Kaya; no functional lymphocytes), Trem2-KO on 5XFAD (Zhou; blunted microglial activation) and oligodendroglial Serpina3n-cKO on cuprizone. Does the C4b⁺ / OL_Dis state survive each?

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../scripts")
import numpy as np, pandas as pd, scipy.sparse as sp, scanpy as sc, seaborn as sns, matplotlib.pyplot as plt
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics.pairwise import cosine_similarity
import oligoc4b_public as pub
sc.settings.verbosity = 0
sc.set_figure_params(dpi=80, frameon=False)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 300)

atlas = sc.read_h5ad(os.path.join(pub.PROCESSED_DIR, "oligodendrocyte_atlas.h5ad"))
for c in ["dataset", "species", "group", "leiden_0.5", "leiden_1.0", "ol_subtype_pred", "scanvi_label"]:
    atlas.obs[c] = atlas.obs[c].astype(str)
print(atlas)
print(atlas.obs.groupby("dataset", observed=True).size())
DATASETS = sorted(atlas.obs["dataset"].unique())
SHORT = {d: d.split("_")[0].replace("2019", "").replace("2020", "").replace("2021", "").replace("2022", "").replace("2023", "").replace("2024", "") for d in DATASETS}
atlas.obs["ds"] = atlas.obs["dataset"].map(SHORT)

## 1. Integration and shared states

In [ ]:
sc.pl.umap(atlas, color=["ds", "species"], ncols=2, size=2, legend_fontsize=7, show=False); plt.show()
sc.pl.umap(atlas, color=["leiden_0.5", "ol_subtype_pred"], ncols=2, size=2, legend_loc="on data", legend_fontsize=7, show=False); plt.show()
sc.pl.umap(atlas, color=["C4b_log", "score_C4b_program", "score_MHCI_IFN", "score_complement_regulators", "ol_dis_prob", "score_myelin"],
           ncols=3, size=2, cmap="magma", vmax="p99", show=False); plt.show()

In [ ]:
# cluster summary: size, dataset mixing (normalised entropy over datasets), C4b+ fraction, scores
def norm_entropy(counts):
    p = counts / counts.sum(); p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(len(counts)))
CL = "leiden_0.5"
rows = []
for cl, sub in atlas.obs.groupby(CL, observed=True):
    comp = sub["dataset"].value_counts().reindex(DATASETS).fillna(0)
    rows.append({"cluster": cl, "n_cells": len(sub), "n_datasets_gt1pct": int((comp / len(sub) > 0.01).sum()), "dataset_entropy": norm_entropy(comp.values),
                 "frac_human": (sub["species"] == "human").mean(), "C4b_pos_frac": sub["C4b_pos"].mean(), "C4b_log_mean": sub["C4b_log"].mean(),
                 "score_C4b_program": sub["score_C4b_program"].mean(), "score_MHCI_IFN": sub["score_MHCI_IFN"].mean(),
                 "score_compl_reg": sub["score_complement_regulators"].mean(), "ol_dis_prob": sub["ol_dis_prob"].mean(), "score_myelin": sub["score_myelin"].mean()})
CLUST = pd.DataFrame(rows).set_index("cluster").sort_values("C4b_pos_frac", ascending=False)
display(CLUST.round(3))
# define C4b-high clusters: top clusters by C4b+ fraction, at least 2x the atlas average and >= 1,000 cells
avg = atlas.obs["C4b_pos"].mean()
C4B_HIGH = CLUST[(CLUST["C4b_pos_frac"] >= 2 * avg) & (CLUST["n_cells"] >= 1000)].index.tolist()
HOMEO = CLUST[(CLUST["C4b_pos_frac"] <= avg) & (CLUST["score_myelin"] >= CLUST["score_myelin"].median()) & (CLUST["n_cells"] >= 1000)].index.tolist()
print(f"atlas C4b+ fraction {avg:.3f}; C4b-high clusters: {C4B_HIGH}; homeostatic reference clusters: {HOMEO}")
atlas.obs["state"] = np.where(atlas.obs[CL].isin(C4B_HIGH), "C4b-high", np.where(atlas.obs[CL].isin(HOMEO), "homeostatic", "other"))

In [ ]:
# which datasets populate the C4b-high clusters, and with which fraction of their cells?
comp = pd.crosstab(atlas.obs["dataset"], atlas.obs[CL], normalize="index")
plt.figure(figsize=(0.5 * comp.shape[1] + 3, 0.4 * comp.shape[0] + 1.5))
sns.heatmap(comp.loc[:, CLUST.index], annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=0.6, cbar_kws={"label": "fraction of the dataset's oligodendrocytes"})
plt.title(f"Cluster composition per dataset (clusters ordered by C4b+ fraction; C4b-high = {C4B_HIGH})"); plt.tight_layout(); plt.show()
# C4b+ fraction inside each cluster, per dataset (is the C4b-high cluster C4b-high in every dataset that contributes to it?)
c4 = atlas.obs[atlas.obs["species"] == "mouse"].groupby(["dataset", CL], observed=True)["C4b_pos"].mean().unstack()[CLUST.index]
print("C4b+ fraction per cluster within each MOUSE dataset (human C4B is not quantifiable):"); display(c4.round(2))

## 2. Label transfer: human OL_Dis subtypes onto every dataset

In [ ]:
ct = pd.crosstab(atlas.obs["dataset"], atlas.obs["ol_subtype_pred"], normalize="index")
display(ct.round(3))
# per-sample fraction of OL_Dis-predicted cells, by group, with a test where >= 3 samples per group
atlas.obs["is_OL_Dis"] = atlas.obs["ol_subtype_pred"].str.startswith("OL_Dis")
rows = []
for d in DATASETS:
    sub = atlas.obs[atlas.obs["dataset"] == d]
    ref = sub["group_ref"].iloc[0]
    per = sub.groupby(["sample", "group"], observed=True).agg(frac_OL_Dis=("is_OL_Dis", "mean"), frac_C4b=("C4b_pos", "mean"), n=("C4b_pos", "size")).reset_index()
    per = per[per["n"] >= 30]
    for g in [x for x in per["group"].unique() if x not in (ref, "unknown", "intermediate")]:
        a, b = per.loc[per.group == g, "frac_OL_Dis"], per.loc[per.group == ref, "frac_OL_Dis"]
        a2, b2 = per.loc[per.group == g, "frac_C4b"], per.loc[per.group == ref, "frac_C4b"]
        p = stats.mannwhitneyu(a, b).pvalue if len(a) >= 3 and len(b) >= 3 else np.nan
        p2 = stats.mannwhitneyu(a2, b2).pvalue if len(a2) >= 3 and len(b2) >= 3 else np.nan
        rows.append({"dataset": d, "contrast": f"{g} vs {ref}", "n_alt": len(a), "n_ref": len(b), "OL_Dis_alt": a.mean(), "OL_Dis_ref": b.mean(), "OL_Dis_p": p,
                     "C4b_alt": a2.mean(), "C4b_ref": b2.mean(), "C4b_p": p2})
TRANSFER = pd.DataFrame(rows)
display(TRANSFER.round(4))
# are C4b+ cells OL_Dis-like? (mouse only)
m = atlas.obs[atlas.obs["species"] == "mouse"]
print("OL_Dis probability in C4b+ vs C4b- mouse oligodendrocytes, per dataset:")
display(m.groupby(["dataset", "C4b_pos"], observed=True)["ol_dis_prob"].mean().unstack().round(3))

## 3. Conserved markers of the C4b-high state (within-dataset DE, then overlap)

In [ ]:
DE = {}
for d in DATASETS:
    sub = atlas[(atlas.obs["dataset"] == d) & atlas.obs["state"].isin(["C4b-high", "homeostatic"])].copy()
    n_hi, n_ho = (sub.obs["state"] == "C4b-high").sum(), (sub.obs["state"] == "homeostatic").sum()
    if n_hi < 50 or n_ho < 50:
        print(f"{d}: skipped (C4b-high {n_hi}, homeostatic {n_ho})"); continue
    sc.tl.rank_genes_groups(sub, "state", groups=["C4b-high"], reference="homeostatic", method="wilcoxon")
    df = sc.get.rank_genes_groups_df(sub, "C4b-high")
    DE[d] = df.set_index("names")
    print(f"{d}: C4b-high {n_hi} vs homeostatic {n_ho}; top-15 up: {', '.join(df.head(15)['names'])}")
TOPN = 100
up_sets = {d: set(df[(df["logfoldchanges"] > 0.5) & (df["pvals_adj"] < 0.01)].head(TOPN).index) for d, df in DE.items()}
count = pd.Series([g for s in up_sets.values() for g in s]).value_counts()
core = count[count >= max(3, len(up_sets) // 2)]
print(f"\ngenes in the top-{TOPN} up-regulated set of >= {max(3, len(up_sets)//2)} of {len(up_sets)} datasets:")
display(core.to_frame("n_datasets").T)
# species split
mouse_ds = [d for d in DE if atlas.obs.loc[atlas.obs.dataset == d, "species"].iloc[0] == "mouse"]
human_ds = [d for d in DE if d not in mouse_ds]
cnt_m = pd.Series([g for d in mouse_ds for g in up_sets[d]]).value_counts()
cnt_h = pd.Series([g for d in human_ds for g in up_sets[d]]).value_counts()
both = sorted(set(cnt_m[cnt_m >= 2].index) & set(cnt_h[cnt_h >= 1].index))
print(f"\nup in >=2 mouse datasets AND >=1 human dataset ({len(both)}):", ", ".join(both))
# where do the complement genes sit?
comp_genes = [g for g in ["C4b", "C4a", "C1qa", "C1qb", "C1qc", "C3", "C3ar1", "Hc", "C5ar1", "C5ar2", "Cfb", "Cd59a", "Cr1l", "Cd55", "Cfh", "Serpina3n", "H2-D1", "B2m"] if any(g in df.index for df in DE.values())]
tab = pd.DataFrame({SHORT[d]: [df.loc[g, "logfoldchanges"] if g in df.index else np.nan for g in comp_genes] for d, df in DE.items()}, index=comp_genes)
plt.figure(figsize=(0.6 * tab.shape[1] + 3, 0.4 * tab.shape[0] + 1.5))
sns.heatmap(tab.astype(float), annot=True, fmt=".1f", cmap="RdBu_r", center=0, vmin=-3, vmax=3, cbar_kws={"label": "log2FC C4b-high vs homeostatic"})
plt.title("Complement genes in the C4b-high vs homeostatic contrast, per dataset"); plt.tight_layout(); plt.show()

## 4. Recurrent gene programs (per-dataset NMF matched across datasets)

In [ ]:
P = pd.DataFrame(atlas.uns["nmf_programs"])
P.index = P.index.astype(str)
# rows = programs "dataset|Pk", columns = genes
S = cosine_similarity(P.values)
Z = linkage(1 - S, method="average")
labels = fcluster(Z, t=0.6, criterion="distance")   # cosine similarity >= 0.4 within a consensus program
prog = pd.DataFrame({"program": P.index, "consensus": labels, "dataset": [i.split("|")[0] for i in P.index]})
cons = prog.groupby("consensus").agg(n_programs=("program", "size"), n_datasets=("dataset", "nunique")).sort_values("n_datasets", ascending=False)
cons = cons[cons["n_datasets"] >= 3]
print(f"{len(cons)} consensus programs recur in >= 3 datasets")
rows = []
for c, r in cons.iterrows():
    members = prog.loc[prog["consensus"] == c, "program"]
    load = P.loc[members].mean(axis=0).sort_values(ascending=False)
    top = list(load.head(25).index)
    flags = [g for g in ["C4b", "C4a", "Serpina3n", "H2-D1", "H2-K1", "B2m", "Cd59a", "Cr1l", "C1qa", "C3", "Klk6", "Apod", "Trf", "Cd9", "Il33"] if g in load.head(150).index]
    rows.append({"consensus": c, "n_datasets": r["n_datasets"], "datasets": ", ".join(sorted({SHORT[d] for d in prog.loc[prog.consensus == c, "dataset"]})),
                 "top_genes": ", ".join(top), "complement/C4b-program genes in top-150": ", ".join(flags)})
CONS = pd.DataFrame(rows).set_index("consensus")
with pd.option_context("display.max_colwidth", 400):
    display(CONS)

## 5. Genetic dependency: does the C4b⁺ / OL_Dis state survive Rag1-KO, Trem2-KO and Serpina3n-cKO?

Per-library values (fraction of oligodendrocytes that are C4b⁺, in the C4b-high clusters, or OL_Dis-predicted; mean MHC-I/IFN score), compared between genotypes within the disease / aged condition. Sample numbers are small, so these are effect sizes with a Mann–Whitney p only where ≥3 vs ≥3 libraries exist.

In [ ]:
def per_sample(mask, group_cols):
    sub = atlas.obs[mask]
    return (sub.groupby(["sample"] + group_cols, observed=True)
               .agg(n=("C4b_pos", "size"), frac_C4b=("C4b_pos", "mean"), frac_C4b_high_cluster=("state", lambda s: (s == "C4b-high").mean()),
                    frac_OL_Dis=("is_OL_Dis", "mean"), MHCI_IFN=("score_MHCI_IFN", "mean"), C4b_program=("score_C4b_program", "mean"))
               .reset_index())
def compare(df, col, a, b, metrics=("frac_C4b", "frac_C4b_high_cluster", "frac_OL_Dis", "MHCI_IFN", "C4b_program")):
    rows = []
    for m in metrics:
        xa, xb = df.loc[df[col] == a, m], df.loc[df[col] == b, m]
        p = stats.mannwhitneyu(xa, xb).pvalue if len(xa) >= 3 and len(xb) >= 3 else np.nan
        rows.append({"metric": m, f"{a} (n={len(xa)})": xa.mean(), f"{b} (n={len(xb)})": xb.mean(), "ratio": xa.mean() / xb.mean() if xb.mean() > 0 else np.nan, "MWU_p": p})
    return pd.DataFrame(rows).set_index("metric")

# (a) Kaya: aged white matter, WT vs Rag1-KO
k = (atlas.obs["dataset"] == "Kaya2022_aged_WM_vs_GM_scRNA") & (atlas.obs["region"].astype(str) == "WM")
if k.sum():
    ps = per_sample(k, ["genotype"]); display(ps.round(3))
    print("Kaya aged white matter: Rag1-KO vs WT"); display(compare(ps, "genotype", "Rag1KO", "WT").round(4))
# (b) Zhou: 5XFAD, Trem2-KO vs Trem2-WT (and non-Tg reference)
z = atlas.obs["dataset"] == "Zhou2020_5XFAD_snRNA"
if z.sum():
    ps = per_sample(z, ["fad", "trem2"]); display(ps.round(3))
    fad = ps[ps["fad"] == "5XFAD"]
    print("Zhou 5XFAD: Trem2-KO vs Trem2-WT"); display(compare(fad, "trem2", "Trem2_KO", "Trem2_WT").round(4))
    print("Zhou non-Tg: Trem2-KO vs Trem2-WT (baseline)"); display(compare(ps[ps["fad"] == "nonTg"], "trem2", "Trem2_KO", "Trem2_WT").round(4))
# (c) Serpina3n-cKO on cuprizone
s3 = atlas.obs["dataset"] == "Serpina3n_Cuprizone_snRNA_mouse"
if s3.sum():
    ps = per_sample(s3, ["diet", "genotype"]); display(ps.round(3))
    cpz = ps[ps["diet"] == "Cuprizone"]
    print("Cuprizone: Serpina3n-cKO vs control genotype"); display(compare(cpz, "genotype", "Serpina3n_cKO", "Ctrl").round(4))

In [ ]:
# visual: per-library fractions for the three contrasts
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (title, mask, hue) in zip(axes, [("Kaya aged WM: WT vs Rag1-KO", k, "genotype"), ("Zhou: 5XFAD x Trem2", z, "trem2"), ("Serpina3n-cKO x cuprizone", s3, "genotype")]):
    if not mask.sum():
        continue
    cols = [hue] + (["fad"] if hue == "trem2" else (["diet"] if "Serpina3n" in title else []))
    ps = per_sample(mask, cols)
    x = "fad" if hue == "trem2" else ("diet" if "Serpina3n" in title else hue)
    sns.stripplot(data=ps, x=x, y="frac_C4b_high_cluster", hue=hue, dodge=True, size=8, ax=ax)
    ax.set_title(title); ax.set_ylabel("fraction in C4b-high clusters"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 6. Summary tables

In [ ]:
print("C4b-high clusters:", C4B_HIGH, "| homeostatic reference:", HOMEO)
summary = atlas.obs.groupby("dataset", observed=True).agg(n=("C4b_pos", "size"), frac_C4b=("C4b_pos", "mean"), frac_C4b_high_cluster=("state", lambda s: (s == "C4b-high").mean()),
                                                          frac_OL_Dis=("is_OL_Dis", "mean"), MHCI_IFN=("score_MHCI_IFN", "mean"))
display(summary.round(3))
display(TRANSFER.round(3))

### Interpretation

*(filled in after execution)*